<h1 align="center"><b> AI Multi-Model Social Media Intelligence System</b></h1>
<h3 align="center"><b></b></h3>

# <h3><b>Project Objective</b></h3>

# This project aims to build an AI-powered system that analyzes social media engagement using multiple techniques including Machine Learning, Natural Language Processing (NLP), Time Series Forecasting, Deep Learning, and Computer Vision. The system will evaluate engagement metrics, analyze sentiment from text, forecast future trends, and present insights in a simple and effective way.

# 1.  import libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from statsmodels.tsa.arima.model import ARIMA
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
import warnings
warnings.filterwarnings('ignore')
import joblib

# 2. Task 1: Social Media Engagement  Data Cleaning & EDA

# 2.1 Load Dataset

In [ ]:
df = pd.read_csv('data/social_media_engagement.csv')

In [ ]:
df.head()

In [ ]:
df.shape

# 2.2 Basic Info

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe().T

# 2.3 Check Missing Values

In [ ]:
df.isnull().sum()

# 2.4 Remove Duplicates

In [ ]:
df.drop_duplicates(inplace=True)

# 2.5 Cap Outliers (IQR Method)

In [ ]:
num_cols = ['likes_count','shares_count','comments_count','impressions','engagement_rate']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    print(f"{col}: outliers capped")

# 2.6 Exploratory Visualizations

In [ ]:
fig = px.histogram(df, x='engagement_rate', nbins=30, title='Engagement Rate Distribution')
fig.show()

In [ ]:
if 'platform' in df.columns:
    fig = px.box(df, x='platform', y='engagement_rate', color='platform', title='Engagement Rate by Platform')
    fig.show()

In [ ]:
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap', color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# 3. Task 2: Feature Engineering

# 3.1 Create Total Engagement

In [ ]:
df['total_engagement'] = df['likes_count'] + df['shares_count'] + df['comments_count']

# 3.2 Extract Time Features (if timestamp exists)

In [ ]:
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.hour
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month

# 3.3 Encode Categorical Columns

In [ ]:
for col in ['platform', 'location']:
    if col in df.columns:
        le = LabelEncoder()
        df[col+'_enc'] = le.fit_transform(df[col].astype(str))

# 4. Task 3: Regression Models (Predict Engagement Rate)

# 4.1 Prepare Features and Target

In [ ]:
feature_cols = [c for c in df.columns if c not in ['timestamp','text_content','engagement_rate','platform','location','language']]
X = df[feature_cols].select_dtypes(include=[np.number])
y = df['engagement_rate']

# 4.2 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4.3 Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4.4 Define and Train Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(n_estimators=100),
    'SVR': SVR()
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    results.append({'Model': name, 'RMSE': rmse, 'R2': r2})
    print(f"{name:20} RMSE: {rmse:.4f}   R2: {r2:.4f}")

# 4.5 Plot Comparison

In [ ]:
res_df = pd.DataFrame(results)
fig = px.bar(res_df, x='Model', y=['RMSE','R2'], barmode='group', title='Model Performance')
fig.show()

# 5. Task 4: NLP Sentiment Analysis (Sentiment140)

# 5.1 Load Dataset

In [ ]:
df_sent = pd.read_csv('data/sentiment140.csv', encoding='latin-1', header=None,
                      names=['target','id','date','flag','user','text'])
df_sent = df_sent[['text','target']]
df_sent['target'] = df_sent['target'].map({0:0, 4:1})  # 0=negative, 1=positive

# 5.2 Sample for Speed

In [ ]:
df_sent = df_sent.sample(20000, random_state=42)

# 5.3 Text Cleaning Function

In [ ]:
def clean_text(text):
    text = re.sub(r'@\w+', '', text)           # remove mentions
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # keep only letters
    return text.lower().strip()

In [ ]:
df_sent['clean'] = df_sent['text'].apply(clean_text)

# 5.4 TF‑IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_sent = tfidf.fit_transform(df_sent['clean'])
y_sent = df_sent['target']

# 5.5 Train Model

In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sent, y_sent, test_size=0.2, random_state=42)
model_sent = LogisticRegression(max_iter=1000)
model_sent.fit(X_train_s, y_train_s)
y_pred_s = model_sent.predict(X_test_s)
print("Sentiment Accuracy:", accuracy_score(y_test_s, y_pred_s))

# 6. Task 5: Toxicity Detection (Jigsaw Toxic Comments)

# 6.1 Load Data

In [ ]:
df_tox = pd.read_csv('data/train.csv')  # from Jigsaw toxic comment dataset

In [ ]:
df_tox.head()

# 6.2 Create Binary Toxic Flag

In [ ]:
toxic_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
df_tox['toxic_binary'] = (df_tox[toxic_cols].sum(axis=1) > 0).astype(int)

# 6.3 Sample and Clean

In [ ]:
df_tox = df_tox.sample(20000, random_state=42)
df_tox['clean'] = df_tox['comment_text'].fillna('').apply(clean_text)

# 6.4 Vectorize and Train

In [ ]:
tfidf_tox = TfidfVectorizer(max_features=5000)
X_tox = tfidf_tox.fit_transform(df_tox['clean'])
y_tox = df_tox['toxic_binary']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_tox, y_tox, test_size=0.2, random_state=42)
model_tox = LogisticRegression(max_iter=1000)
model_tox.fit(X_train_t, y_train_t)
y_pred_t = model_tox.predict(X_test_t)
print("Toxicity Accuracy:", accuracy_score(y_test_t, y_pred_t))

# 7. Task 6: Time Series Forecasting (Air Passengers)

# 7.1 Load Data

In [ ]:
df_air = pd.read_csv('data/AirPassengers.csv')
df_air.columns = ['Month','Passengers']
df_air['Month'] = pd.to_datetime(df_air['Month'])
df_air.set_index('Month', inplace=True)

# 7.2 Plot Original Series

In [ ]:
fig = px.line(df_air, y='Passengers', title='Monthly Air Passengers')
fig.show()

# 7.3 Train‑Test Split

In [ ]:
train = df_air[:100]
test = df_air[100:]

# 7.4 Fit ARIMA Model

In [ ]:
model = ARIMA(train, order=(5,1,0))
model_fit = model.fit()
forecast = model_fit.forecast(steps=len(test))

# 7.5 Plot Forecast vs Actual

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train['Passengers'], name='Train'))
fig.add_trace(go.Scatter(x=test.index, y=test['Passengers'], name='Actual'))
fig.add_trace(go.Scatter(x=test.index, y=forecast, name='Forecast', line=dict(dash='dash')))
fig.update_layout(title='ARIMA Forecast of Air Passengers')
fig.show()

# 8. Task 7: Clustering (Customer Segmentation)

# 8.1 Load Data

In [ ]:
df_cust = pd.read_csv('data/Mall_Customers.csv')

In [ ]:
df_cust.head()

# 8.2 Select Features

In [ ]:
X_cust = df_cust[['Annual Income (k$)', 'Spending Score (1-100)']]

# 8.3 Scale Features

In [ ]:
scaler_cust = StandardScaler()
X_cust_scaled = scaler_cust.fit_transform(X_cust)

# 8.4 Elbow Method

In [ ]:
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_cust_scaled)
    inertia.append(km.inertia_)
fig = px.line(x=range(1,11), y=inertia, markers=True, title='Elbow Method')
fig.show()

# 8.5 Apply K‑Means (k=5)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X_cust_scaled)
df_cust['Cluster'] = clusters

# 8.6 Visualize Clusters

In [ ]:
fig = px.scatter(df_cust, x='Annual Income (k$)', y='Spending Score (1-100)',
                 color='Cluster', title='Customer Segments')
fig.show()

# 9. Task 8: Deep Learning (Pima Indians Diabetes)

# 9.1 Load Data

In [ ]:
df_diab = pd.read_csv('data/diabetes.csv')
X = df_diab.drop('Outcome', axis=1)
y = df_diab['Outcome']

# 9.2 Split and Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 9.3 Build Neural Network

In [ ]:
model_nn = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## 9.4 Train

In [ ]:
history = model_nn.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# 9.5 Evaluate

In [ ]:
loss, acc = model_nn.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

# 9.6 Plot Training History

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss'))
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss'))
fig.update_layout(title='Loss over Epochs')
fig.show()

# 10. Task 9: Computer Vision – CNN (Chest X‑Ray Pneumonia)

# 10.1 Data Generators (Adjust Paths)

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_generator = train_datagen.flow_from_directory(
    'data/chest_xray/train',          # path to training images
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    subset='training'
)
val_generator = train_datagen.flow_from_directory(
    'data/chest_xray/train',
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

# 10.2 Build CNN Model

In [ ]:
model_cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])
model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 10.3 Train

In [ ]:
history_cnn = model_cnn.fit(train_generator, validation_data=val_generator, epochs=5)
print("CNN code ready – uncomment when dataset is available.")

# 11. Task 10: Transfer Learning (Flowers Recognition)

# 11.1 Load Pre‑trained VGG16

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150,150,3))
base_model.trainable = False

# 11.2 Build Classifier on Top

In [ ]:
model_transfer = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(5, activation='softmax')   # 5 flower categories
])
model_transfer.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 11.3 Data Generators for Flowers (Adjust Paths)

In [ ]:
flower_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_flowers = flower_datagen.flow_from_directory(
    'data/flowers/train',
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
val_flowers = flower_datagen.flow_from_directory(
    'data/flowers/train',
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# 11.4 Train

In [ ]:
history_transfer = model_transfer.fit(train_flowers, validation_data=val_flowers, epochs=5)
print("Transfer learning code ready – uncomment when dataset is available.")

# 12. Save Models

In [ ]:
# Save best regression model
joblib.dump(models['Random Forest'], 'models/best_engagement_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# Save sentiment and toxicity models
joblib.dump(model_sent, 'models/sentiment_model.pkl')
joblib.dump(tfidf, 'models/tfidf_sentiment.pkl')
joblib.dump(model_tox, 'models/toxicity_model.pkl')

# Save neural network
model_nn.save('models/diabetes_nn.h5')

print("All models saved to 'models/' folder.")

# 1. Predict Social Media Engagement Rate (Regression)

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the best regression model and scaler
model = joblib.load('models/best_engagement_model.pkl')
scaler = joblib.load('models/scaler.pkl')
feature_cols = joblib.load('models/feature_cols.pkl')  # saved earlier

print("\n Predict Engagement Rate")
print("Enter the following features:")

# Collect input (adjust according to your feature list)
# For simplicity, we assume all features are numeric and available
input_data = {}
for col in feature_cols:
    val = float(input(f"{col}: "))
    input_data[col] = val

# Convert to DataFrame and scale
input_df = pd.DataFrame([input_data])
input_scaled = scaler.transform(input_df)

# Predict
pred = model.predict(input_scaled)[0]
print(f"\n Predicted Engagement Rate: {pred:.4f}")

# 2. Predict Sentiment (Positive/Negative) from Text

In [ ]:
import joblib
import re

# Load sentiment model and vectorizer
model = joblib.load('models/sentiment_model.pkl')
tfidf = joblib.load('models/tfidf_sentiment.pkl')

def clean_text(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

print("\n💬 Sentiment Analysis")
text = input("Enter your text: ")
cleaned = clean_text(text)
vec = tfidf.transform([cleaned])
pred = model.predict(vec)[0]
prob = model.predict_proba(vec)[0]

if pred == 1:
    print(f"✅ Positive sentiment (confidence {prob[1]:.2f})")
else:
    print(f"❌ Negative sentiment (confidence {prob[0]:.2f})")

# 3. Predict Toxicity (Toxic / Non‑Toxic)

In [ ]:
import joblib
import re

# Load toxicity model and vectorizer
model = joblib.load('models/toxicity_model.pkl')
tfidf_tox = joblib.load('models/tfidf_toxicity.pkl')

def clean_text(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

print("\n Toxicity Detection")
text = input("Enter a comment: ")
cleaned = clean_text(text)
vec = tfidf_tox.transform([cleaned])
pred = model.predict(vec)[0]
prob = model.predict_proba(vec)[0]

if pred == 1:
    print(f"⚠️ Toxic content (probability {prob[1]:.2f})")
else:
    print(f"✅ Non‑toxic (probability {prob[0]:.2f})")

# 4. Forecast Air Passengers (Next Months)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import plotly.graph_objects as go

# Load the pre‑trained model (or train it now)
# For demonstration, we re‑train on the full dataset
df_air = pd.read_csv('data/AirPassengers.csv')
df_air.columns = ['Month','Passengers']
df_air['Month'] = pd.to_datetime(df_air['Month'])
df_air.set_index('Month', inplace=True)

model = ARIMA(df_air, order=(5,1,0))
model_fit = model.fit()

print("\n📈 Air Passenger Forecast")
months = int(input("How many months ahead to forecast? "))
forecast = model_fit.forecast(steps=months)
print("Forecasted passengers:")
for i, val in enumerate(forecast, 1):
    print(f"Month {i}: {val:.0f}")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_air.index, y=df_air['Passengers'], name='Historical'))
future_dates = pd.date_range(df_air.index[-1], periods=months+1, freq='MS')[1:]
fig.add_trace(go.Scatter(x=future_dates, y=forecast, name='Forecast', line=dict(dash='dash')))
fig.update_layout(title='Air Passenger Forecast')
fig.show()

# 5. Predict Customer Segment (Cluster)

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the trained k‑means model and scaler
kmeans = joblib.load('models/kmeans.pkl')
scaler = joblib.load('models/cluster_scaler.pkl')

print("\n👥 Customer Segment Prediction")
income = float(input("Annual Income (k$): "))
spending = float(input("Spending Score (1-100): "))

input_data = pd.DataFrame([[income, spending]], columns=['Annual Income (k$)', 'Spending Score (1-100)'])
scaled = scaler.transform(input_data)
cluster = kmeans.predict(scaled)[0]

print(f"\n🔮 Predicted Cluster: {cluster}")
# Optional: add cluster description (from your earlier analysis)
cluster_desc = {
    0: "Low income, low spending",
    1: "Low income, high spending",
    2: "Medium income, medium spending",
    3: "High income, low spending",
    4: "High income, high spending"
}
print(f"Segment: {cluster_desc.get(cluster, 'Unknown')}")

# 6. Predict Diabetes (Deep Learning)

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import joblib

# Load model and scaler
model = tf.keras.models.load_model('models/diabetes_nn.h5')
scaler = joblib.load('models/diabetes_scaler.pkl')
feature_names = joblib.load('models/diabetes_features.pkl')  # list of feature names

print("\n🩺 Diabetes Prediction")
print("Enter the following values:")

input_data = {}
for name in feature_names:
    val = float(input(f"{name}: "))
    input_data[name] = val

input_df = pd.DataFrame([input_data])
scaled = scaler.transform(input_df)
pred_prob = model.predict(scaled)[0][0]

if pred_prob >= 0.5:
    print(f"⚠️ High risk of diabetes (probability {pred_prob:.2f})")
else:
    print(f"✅ Low risk of diabetes (probability {1-pred_prob:.2f})")

# 7. Predict Pneumonia from Chest X‑Ray Image (CNN)

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# Load pre‑trained CNN model
model = tf.keras.models.load_model('models/pneumonia_cnn.h5')  # if saved

print("\n🩻 Pneumonia Detection from Chest X‑Ray")
img_path = input("Enter path to chest X‑ray image: ")

img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)[0][0]
if pred >= 0.5:
    print(f"⚠️ Pneumonia detected (confidence {pred:.2f})")
else:
    print(f"✅ Normal (confidence {1-pred:.2f})")

# 8. Predict Flower Type (Transfer Learning)

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# Load pre‑trained transfer learning model
model = tf.keras.models.load_model('models/flowers_transfer.h5')
class_names = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']  # adjust

print("\n Flower Classification")
img_path = input("Enter path to flower image: ")

img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = np.argmax(pred)
confidence = np.max(pred)

print(f"🌼 Predicted flower: {class_names[pred_class]} (confidence {confidence:.2f})")